# 22.5 Docker 入门:把模型服务打包成"到处能跑" / Docker Basics: package your service to "run anywhere"

**中文**:22.3 我们写好了 FastAPI 模型服务,它在**你的机器上**跑得好好的。但部署到服务器上,常常就崩了:Python 版本不对、缺个系统库、依赖版本冲突、环境变量没配……这就是臭名昭著的**"在我机器上明明能跑"(works on my machine)** 问题。**Docker** 是它的终极解药:*把你的应用 + 所有依赖(Python 包、系统库)+ 运行环境,一起打包成一个自包含的"镜像(image)",这个镜像在任何装了 Docker 的机器上都以完全相同的方式运行*。它是现代软件部署的**事实标准**,也是 MLOps 从"我这能跑"到"哪都能跑"的关键一环。本节讲清 Docker 的核心概念(镜像/容器/分层),并从零实现一个**镜像分层缓存**模拟器,让你亲手看懂 Dockerfile 里最重要的优化——**为什么指令顺序决定构建速度**。
**English**: In 22.3 we built a FastAPI model service that runs fine **on your machine**. But deployed to a server it often breaks: wrong Python version, a missing system library, dependency conflicts, unset environment variables… the infamous **"works on my machine"** problem. **Docker** is its ultimate cure: *package your app + all dependencies (Python packages, system libraries) + runtime environment into a self-contained "image," which runs identically on any machine with Docker installed*. It's the **de-facto standard** for modern software deployment and MLOps's key link from "runs here" to "runs anywhere." This section explains Docker's core concepts (image/container/layers) and builds an **image layer-cache** simulator from scratch, so you see the most important Dockerfile optimization firsthand — **why instruction order determines build speed**.

---

**中文**:**核心概念(用类比秒懂)**:
**English**: **Core concepts (an analogy makes it instant)**:
- **中文**:**镜像(image)= 类,容器(container)= 实例**。镜像是一个**只读的模板**(打包好的应用+环境),容器是镜像**运行起来的一个实例**。一个镜像可以启动很多个容器(就像一个类可以 new 很多对象)。
  **Image = class, container = instance**. An image is a **read-only template** (packaged app + environment); a container is a **running instance** of an image. One image can start many containers (like one class can instantiate many objects).
- **中文**:**Dockerfile = 构建镜像的配方**。一行行指令(用哪个基础镜像、装什么依赖、拷贝什么代码、启动命令),`docker build` 按它造出镜像。
  **Dockerfile = the recipe to build an image**. Line-by-line instructions (which base image, what dependencies to install, what code to copy, the startup command); `docker build` produces the image from it.
- **中文**:**分层(layers)是 Docker 的精髓**。Dockerfile 的**每一条指令生成一个只读层**,镜像就是这些层叠起来。关键:**构建时,没变的层直接用缓存,只有从第一个变化的层往下才重新构建**。所以指令顺序极其重要——把"很少变的"(装依赖)放前面、"经常变的"(拷贝代码)放后面,能让每次改代码后的重建快几十倍。
  **Layers are Docker's essence**. **Each Dockerfile instruction creates a read-only layer**, and the image is these layers stacked. Key: **on build, unchanged layers use the cache, and only from the first changed layer down is rebuilt**. So instruction order matters enormously — put "rarely changing" (install dependencies) first and "frequently changing" (copy code) last, making rebuilds after code changes tens of times faster.
- **中文**:**镜像仓库(registry)**:Docker Hub / 云厂商的仓库,`push`/`pull` 镜像,像 GitHub 之于代码。
  **Registry**: Docker Hub / cloud registries; `push`/`pull` images, like GitHub for code.

> 💡 **面试速查 / Interview cheat-sheet（★★ 部署必考）**
> **中文**:**Docker** 解决"在我机器上能跑"——把应用+依赖+环境打包成**可移植镜像**, 到处一致运行。**镜像(只读模板)vs 容器(运行实例)**;**Dockerfile**=构建配方;**每条指令=一层**, 构建时未变的层走缓存、从第一个变化层往下重建→**指令顺序决定构建速度**(先 COPY requirements+install, 后 COPY 代码, 否则每次改代码都重装依赖)。**关键优化**:①层缓存排序(依赖在前代码在后)②**多阶段构建**(build 阶段编译, 只拷产物到 slim 运行镜像→镜像小)③用 slim/distroless 基础镜像④`.dockerignore` 排除无关文件⑤**别以 root 运行**、别把密钥打进镜像。**vs 虚拟机**:容器共享宿主内核(轻量秒启), VM 带整个 OS(重)。**docker-compose**=多容器编排(如 app+db+redis)。**registry**=Docker Hub/ECR/GCR push-pull。ML 特有:镜像常很大(CUDA/torch), 用多阶段+层缓存优化; 模型权重可打进镜像或挂载卷/对象存储。面试金句:*"Docker 把应用和全部依赖打包成可移植镜像解决环境不一致; 镜像是只读模板、容器是运行实例; 每条 Dockerfile 指令是一层, 未变层走缓存, 所以把依赖安装放代码拷贝之前能让重建快几十倍; 生产用多阶段构建减小镜像、slim 基础镜像、非 root 用户、docker-compose 编排多容器, 再上 K8s。"*
> **English**: **Docker** solves "works on my machine" — package app + dependencies + environment into a **portable image** that runs identically everywhere. **Image (read-only template) vs container (running instance)**; **Dockerfile** = build recipe; **each instruction = a layer**, and on build unchanged layers use the cache while everything from the first changed layer down rebuilds → **instruction order determines build speed** (COPY requirements + install first, COPY code last, else every code change reinstalls dependencies). **Key optimizations**: ① layer-cache ordering (dependencies before code) ② **multi-stage builds** (compile in a build stage, copy only artifacts to a slim runtime image → small image) ③ use slim/distroless base images ④ `.dockerignore` to exclude irrelevant files ⑤ **don't run as root**, don't bake secrets into the image. **vs VMs**: containers share the host kernel (lightweight, seconds to start), VMs carry a whole OS (heavy). **docker-compose** = multi-container orchestration (e.g. app + db + redis). **Registry** = Docker Hub/ECR/GCR push-pull. ML-specific: images are often large (CUDA/torch), optimize with multi-stage + layer caching; model weights can be baked into the image or mounted via volumes/object storage. Interview line: *"Docker packages an app and all dependencies into a portable image to solve environment inconsistency; the image is a read-only template, the container a running instance; each Dockerfile instruction is a layer, unchanged layers use the cache, so putting dependency installs before code copies makes rebuilds tens of times faster; production uses multi-stage builds to shrink images, slim base images, a non-root user, docker-compose for multi-container, then K8s."*


In [ ]:

# ============================================================
# 从零实现镜像分层缓存:为什么 Dockerfile 指令顺序决定构建速度 / layer-cache: why instruction order matters
# 中文:模拟 docker build。每条指令生成一个"层", 层的哈希 = f(父层哈希, 指令, 影响该层的内容)。
#      未变的层(哈希命中缓存)直接跳过=快; 变了就从此层往下全部重建=慢。
# English: simulate docker build. Each instruction makes a "layer" whose hash = f(parent hash, instruction, content).
#      Unchanged layers (hash hits cache) are skipped = fast; a change rebuilds everything from that layer down = slow.
# ============================================================
import hashlib
def docker_build(instructions, cache):
    parent="" ; result=[]
    for name, content in instructions:                     # content = 会影响这一层的东西(如 requirements 内容)/ what affects this layer
        key=hashlib.md5((parent+name+content).encode()).hexdigest()[:8]
        if key in cache: result.append((name, "CACHED ✓ 缓存(跳过)"))    # 缓存命中→跳过→快 / cache hit → skip → fast
        else: cache.add(key); result.append((name, "BUILT  ⏳ 重建(慢)"))  # 缓存未命中→执行→慢 / miss → run → slow
        parent=key                                          # 下一层依赖上一层的哈希 / next layer depends on this hash
    return result

# ✓ 好的排序:先拷 requirements + 装依赖, 最后才拷代码 / GOOD: copy requirements + install FIRST, copy code LAST
def good(code, reqs): return [("FROM python:3.11-slim",""), ("COPY requirements.txt", reqs),
                              ("RUN pip install -r requirements.txt", reqs), ("COPY . .", code),
                              ("CMD uvicorn app:app", "")]
# ✗ 坏的排序:先把所有东西一起拷, 再装依赖 / BAD: copy everything first, then install
def bad(code, reqs):  return [("FROM python:3.11-slim",""), ("COPY . .", code+reqs),
                              ("RUN pip install -r requirements.txt", reqs), ("CMD uvicorn app:app", "")]

print("=== ✓ 好排序(依赖层在代码层之前)===")
cache=set(); [print(f"  首次: {n:36} {s}") for n,s in docker_build(good("code_v1","reqs_v1"), cache)]
print("  --- 只改了应用代码, 依赖没变, 重新 build: ---")
[print(f"  重建: {n:36} {s}") for n,s in docker_build(good("code_v2","reqs_v1"), cache)]

print("\n=== ✗ 坏排序(先 COPY . . 再装依赖)===")
cache=set(); [print(f"  首次: {n:36} {s}") for n,s in docker_build(bad("code_v1","reqs_v1"), cache)]
print("  --- 只改了应用代码, 依赖没变, 重新 build: ---")
[print(f"  重建: {n:36} {s}") for n,s in docker_build(bad("code_v2","reqs_v1"), cache)]
print("\n关键:好排序下改代码时 pip install 仍走缓存(秒建); 坏排序下改一行代码就重装全部依赖(慢几十倍)")


**中文**:上面模拟了缓存机制。下面是给 22.3 那个 FastAPI 模型服务的**真实 Dockerfile**(注意指令顺序):
**English**: The above simulates the cache mechanism. Below is a **real Dockerfile** for 22.3's FastAPI model service (note the instruction order):

```dockerfile
# Dockerfile —— 构建 / build: docker build -t iris-api .   运行 / run: docker run -p 8000:8000 iris-api
FROM python:3.11-slim                       # 轻量基础镜像(不是完整 Ubuntu)/ slim base image

WORKDIR /app

# ★ 先拷依赖清单 + 安装 —— 这一层只在 requirements 变化时才重建 / deps layer, rebuilt only when requirements change
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# 再拷应用代码 —— 改代码只重建这一层, 上面的 pip install 层走缓存 / copy code last, deps stay cached on code change
COPY . .

# 安全:创建非 root 用户运行 / security: run as a non-root user
RUN useradd -m appuser
USER appuser

EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
```

**中文**:配套的 `docker-compose.yml`(多容器编排:模型服务 + Redis 缓存):
**English**: A companion `docker-compose.yml` (multi-container: model service + Redis cache):

```yaml
# docker-compose.yml —— 一键起多个容器 / one command starts multiple containers: docker compose up
services:
  api:
    build: .                    # 用当前目录的 Dockerfile 构建 / build from the local Dockerfile
    ports: ["8000:8000"]        # 宿主:容器 端口映射 / host:container port mapping
    environment: [MODEL_PATH=/app/iris_model.joblib]
    depends_on: [redis]         # 等 redis 先起 / wait for redis
  redis:
    image: redis:7-alpine       # 直接用现成镜像 / use a prebuilt image
```


In [ ]:

# ============================================================
# 可视化:分层与缓存失效 / layer stack & cache invalidation
# ============================================================
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(14,5.5))
for col,(title,layers,changed) in enumerate([
    ("✓ 好排序:改代码只重建1层",["FROM slim","COPY requirements","RUN pip install","COPY . . (代码)","CMD"],3),
    ("✗ 坏排序:改代码重建3层",["FROM slim","COPY . . (代码+依赖)","RUN pip install","CMD"],1)]):
    a=ax[col]; a.axis("off"); a.set_title(title,fontsize=12,weight="bold")
    n=len(layers)
    for i,lab in enumerate(layers):
        cached = i < changed
        c="#55A868" if cached else "#C44E52"
        a.add_patch(plt.Rectangle((0.2,0.8-i*0.16),0.6,0.13,fc=c,alpha=0.35,ec=c,transform=a.transAxes))
        tag="缓存✓" if cached else "重建⏳"
        a.text(0.5,0.865-i*0.16,f"{lab}   [{tag}]",ha="center",va="center",fontsize=9,transform=a.transAxes)
    a.text(0.5,0.04,"绿=命中缓存(快), 红=需重建(慢)。改代码时红色层越少越好",ha="center",fontsize=8,style="italic",transform=a.transAxes)
plt.tight_layout(); plt.savefig("/tmp/mlops05_viz.png",dpi=80); plt.show()
print("好排序:改代码只重建'COPY 代码'+'CMD'两层, pip install 走缓存; 坏排序:改代码把 pip install 也拖下水重装")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **Docker 的价值是"消灭环境差异",这是可复现性的基石**:"在我机器上能跑"是所有软件工程师和数据科学家都经历过的噩梦——你的代码依赖某个特定版本的库、某个系统组件、某个环境变量,换台机器就崩。Docker 把**应用 + 全部依赖 + 运行环境**冻结进一个镜像,让"我这能跑"直接等于"哪都能跑"。对 ML 尤其重要:模型服务往往依赖一长串精确版本的库(还记得 22.1 的版本漂移吗?),Docker 把这份环境**钉死并可复现**——这也是为什么现代部署几乎都从"打个镜像"开始。
2. **分层缓存是 Docker 最实用、也最能体现功力的优化**:我们的模拟器清楚地показал:**Dockerfile 指令的顺序,直接决定你每次改代码后要等 3 秒还是 3 分钟**。核心原则一句话——**把"很少变的"放前面,"经常变的"放后面**。依赖(requirements)很少变、装起来很慢,所以 `COPY requirements.txt` + `pip install` 要放在 `COPY . .`(应用代码,经常变)之前。这样改代码时,昂贵的 `pip install` 层走缓存、秒过;而新手常犯的"先 `COPY . .` 再装依赖",会让每改一行代码都把所有依赖重装一遍。这是一个**看起来微不足道、实则每天影响开发效率**的细节,面试中问到能立刻区分"用过 Docker"和"懂 Docker"。
3. **诚实的边界与进阶**:①**镜像大小是 ML 的老大难**——一个带 CUDA + PyTorch 的镜像动辄好几 GB,拖慢构建、推送、拉取、冷启动。解药是**多阶段构建(multi-stage build)**:在"构建阶段"编译/安装,只把最终产物拷进一个干净的 slim 运行镜像,把编译工具链等中间物统统甩掉;再配 slim/distroless 基础镜像、`.dockerignore`。②**容器不是虚拟机**——容器共享宿主机内核(所以轻量、秒启),VM 各自带完整 OS(重)。别把容器当"小虚拟机",它是"隔离的进程"。③**安全**:别用 root 运行容器、别把密钥/凭证打进镜像(用环境变量或 secret 管理)、及时更新基础镜像补漏洞。④**Docker 只是打包和单机运行**——真正的生产要**编排**(多副本、扩缩、自愈、滚动更新),那是 **Kubernetes(22.6)** 的事;Docker 打的镜像正是 K8s 调度的基本单位。**结论:Docker 把"环境"变成可版本化、可复现、可移植的产物,是现代部署的地基;掌握分层缓存等优化让你从'会用'到'用好',而它打出的镜像正是下一节 K8s 编排的原材料。**

**English**:
1. **Docker's value is "eliminating environment differences," the cornerstone of reproducibility**: "works on my machine" is a nightmare every software engineer and data scientist has lived — your code depends on a specific library version, a system component, an environment variable, and it breaks on another machine. Docker freezes **app + all dependencies + runtime environment** into an image, making "runs here" equal "runs anywhere." Especially crucial for ML: model services often depend on a long list of precisely-versioned libraries (remember 22.1's version drift?), and Docker **pins and reproduces** this environment — which is why modern deployment almost always starts with "build an image."
2. **Layer caching is Docker's most practical optimization and the clearest sign of skill**: our simulator clearly showed that **Dockerfile instruction order directly determines whether each code change costs you 3 seconds or 3 minutes**. The core principle in one sentence — **put "rarely changing" first, "frequently changing" last**. Dependencies (requirements) rarely change and are slow to install, so `COPY requirements.txt` + `pip install` must come before `COPY . .` (app code, frequently changing). Then code changes let the expensive `pip install` layer use the cache and pass instantly; the beginner mistake of "`COPY . .` first, then install" reinstalls all dependencies on every code line change. This is a **seemingly trivial detail that actually affects daily development efficiency**, and in interviews it instantly distinguishes "has used Docker" from "understands Docker."
3. **Honest limits and advanced points**: ① **Image size is a perennial ML headache** — a CUDA + PyTorch image is easily several GB, slowing builds, pushes, pulls, cold starts. The cure is **multi-stage builds**: compile/install in a "build stage" and copy only the final artifacts into a clean slim runtime image, discarding compiler toolchains and intermediates; plus slim/distroless base images and `.dockerignore`. ② **Containers aren't VMs** — containers share the host kernel (hence lightweight, seconds to start), VMs each carry a full OS (heavy). Don't treat a container as a "small VM"; it's an "isolated process." ③ **Security**: don't run containers as root, don't bake secrets/credentials into images (use environment variables or secret management), promptly update base images to patch vulnerabilities. ④ **Docker only packages and runs on one machine** — real production needs **orchestration** (multiple replicas, scaling, self-healing, rolling updates), which is **Kubernetes (22.6)**'s job; the image Docker builds is exactly K8s's basic scheduling unit. **Conclusion: Docker turns "the environment" into a versionable, reproducible, portable artifact, the foundation of modern deployment; mastering optimizations like layer caching takes you from "can use" to "uses well," and the image it builds is the raw material for the next section's K8s orchestration.**

> 💼 **实战视角 / Practical angle**
> **中文**:Docker 落地:①**Dockerfile 排序**:`COPY requirements.txt`+`pip install` 在 `COPY . .` 之前(层缓存, 改代码秒建);②**多阶段构建**减小镜像(build 阶段装编译依赖, 运行阶段只拷产物到 slim 镜像)——ML 镜像尤其需要;③`.dockerignore` 排除 `.git`/数据/虚拟环境;④**非 root 用户**运行、别打密钥进镜像(用 env/secret);⑤**docker-compose** 本地起多容器(app+db+cache);�6镜像推到 registry(Docker Hub/ECR/GCR), 打语义化 tag;⑦ML 大镜像:固定 CUDA/torch 版本、用官方 devel/runtime 镜像分层。**调试**:`docker logs`、`docker exec -it 容器 bash` 进容器排查。面试金句:*"Docker 把应用+依赖+环境打包成可移植镜像解决环境不一致; 每条 Dockerfile 指令是一层, 依赖安装放代码拷贝之前利用层缓存让重建快几十倍; 用多阶段构建和 slim 基础镜像减小 ML 镜像、非 root 运行、compose 编排多容器; 镜像是 K8s 调度的基本单位。"*
> **English**: Docker in practice: ① **Dockerfile ordering**: `COPY requirements.txt` + `pip install` before `COPY . .` (layer cache, instant rebuild on code change); ② **multi-stage builds** to shrink images (install build deps in a build stage, copy only artifacts to a slim runtime image) — especially needed for ML images; ③ `.dockerignore` to exclude `.git`/data/virtualenv; ④ run as a **non-root user**, don't bake secrets into images (use env/secret); ⑤ **docker-compose** to run multiple containers locally (app+db+cache); ⑥ push images to a registry (Docker Hub/ECR/GCR) with semantic tags; ⑦ large ML images: pin CUDA/torch versions, use official devel/runtime images in layers. **Debug**: `docker logs`, `docker exec -it container bash` to inspect inside. Interview line: *"Docker packages app + dependencies + environment into a portable image to solve environment inconsistency; each Dockerfile instruction is a layer, and putting dependency installs before code copies exploits the layer cache to make rebuilds tens of times faster; use multi-stage builds and slim base images to shrink ML images, run non-root, compose for multi-container; the image is K8s's basic scheduling unit."*

---
### 小结 / Summary
- **中文**:Docker 把应用+依赖+环境打包成可移植镜像, 解决"在我机器上能跑"; 镜像=模板, 容器=运行实例。
- **English**: Docker packages app + dependencies + environment into a portable image, solving "works on my machine"; image = template, container = running instance.
- **中文**:每条指令=一层, 未变层走缓存→指令顺序决定构建速度(依赖装在代码拷贝之前, 重建快几十倍)。
- **English**: Each instruction = a layer, unchanged layers use the cache → instruction order determines build speed (install deps before copying code, tens of times faster rebuilds).
- **中文**:多阶段构建+slim 镜像减小体积、非 root、别打密钥; 容器共享内核比 VM 轻; 镜像是 K8s(22.6)调度的基本单位。
- **English**: Multi-stage builds + slim images shrink size, non-root, no baked secrets; containers share the kernel (lighter than VMs); the image is K8s's (22.6) basic scheduling unit.
